# TRT-LLM Benchmark Results — Llama 3.1 70B

Compares roofline estimator predictions against TRT-LLM gptManagerBenchmark (static batching) measurements.

Set `WORKLOAD` and `INSTANCE` below.

In [ ]:
import json
import glob
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

# ─── Configuration ─────────────────────────────────────────────────
WORKLOAD = "in763-out232"
INSTANCE = "g6-48xlarge"   # subdirectory name in measured/
BENCH_TOOL = "gptBench"    # gptBench or trtllm-bench
# ───────────────────────────────────────────────────────────────────

BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
PRED_DIR = os.path.join(BASE_DIR, WORKLOAD, "predicted")
MEAS_DIR = os.path.join(BASE_DIR, WORKLOAD, "measured", INSTANCE, BENCH_TOOL, "log")

INSTANCE_MAP = {
    "g5-48xlarge": "g5.48xlarge",
    "g6-48xlarge": "g6.48xlarge",
    "g6e-48xlarge": "g6e.48xlarge",
    "p4d-24xlarge": "p4d.24xlarge",
    "p5-48xlarge": "p5.48xlarge",
}

COLS = ["ttft_ms", "tpot_ms", "e2e_ms", "rps"]
JOIN_KEYS = ["tp", "pp", "batch"]

print(f"Workload: {WORKLOAD}")
print(f"Instance: {INSTANCE}")
print(f"Predicted dir: {PRED_DIR} ({len(glob.glob(os.path.join(PRED_DIR, 'est_*.json')))} files)")
print(f"Measured dir: {MEAS_DIR} ({len(glob.glob(os.path.join(MEAS_DIR, '*.log')))} logs)")

## Helper Functions

In [ ]:
def parse_gptbench_log(filepath: str) -> dict:
    """Parse gptManagerBenchmark log file into a dict of metrics."""
    metrics = {}
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line.startswith("[BENCHMARK]"):
                continue
            parts = line.replace("[BENCHMARK] ", "").strip()
            tokens = parts.rsplit(" ", 1)
            if len(tokens) != 2:
                continue
            key_raw, val_str = tokens
            key = re.sub(r'\(.*?\)', '', key_raw).strip()
            try:
                metrics[key] = float(val_str)
            except ValueError:
                metrics[key] = val_str
    return metrics


def parse_log_filename(filename: str) -> dict:
    """Extract tp, pp, bs from filename like trtllm_tp8_pp1_bs16.log"""
    m = re.match(r'trtllm_tp(\d+)_pp(\d+)_bs(\d+)\.log', filename)
    if m:
        return {"tp": int(m.group(1)), "pp": int(m.group(2)), "batch": int(m.group(3))}
    return None


def load_measured(meas_dir: str) -> pd.DataFrame:
    """Load all gptManagerBenchmark log files into a DataFrame."""
    rows = []
    for filepath in sorted(glob.glob(os.path.join(meas_dir, "trtllm_*.log"))):
        filename = os.path.basename(filepath)
        info = parse_log_filename(filename)
        if info is None:
            continue
        metrics = parse_gptbench_log(filepath)
        if not metrics or metrics.get("num_samples", 0) == 0:
            continue
        
        ttft = metrics.get("avg_time_to_first_token", 0)
        e2e = metrics.get("avg_sequence_latency", 0)
        itl = metrics.get("avg_inter_token_latency", 0)
        
        rows.append({
            "tp": info["tp"],
            "pp": info["pp"],
            "batch": info["batch"],
            "ttft_ms": round(ttft, 2),
            "tpot_ms": round(itl, 4),
            "e2e_ms": round(e2e, 2),
            "rps": round(metrics.get("seq_throughput", 0), 4),
            "token_throughput": round(metrics.get("token_throughput", 0), 2),
            "num_samples": int(metrics.get("num_samples", 0)),
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(JOIN_KEYS).reset_index(drop=True)
    return df


def load_predicted(pred_dir: str, instance_filter: str = None) -> pd.DataFrame:
    """Load estimator prediction JSONs."""
    files = sorted(glob.glob(os.path.join(pred_dir, "est_*.json")))
    rows = []
    for f in files:
        with open(f) as fp:
            d = json.load(fp)
        if not d.get("feasible", False):
            continue
        inst = d["instance_type"]
        if instance_filter:
            norm_inst = inst.replace(".", "-")
            norm_filter = instance_filter.replace(".", "-")
            if norm_inst != norm_filter:
                continue
        for entry in d.get("batch_sweep", []):
            rows.append({
                "tp": d["tp_size"],
                "pp": d["pp_size"],
                "batch": entry["batch_size"],
                "ttft_ms": round(entry.get("ttft_ms", 0), 2),
                "tpot_ms": round(entry.get("tpot_ms", 0), 4),
                "e2e_ms": round(entry["batch_latency_ms"], 2),
                "rps": round(entry["throughput_rps"], 4),
            })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(JOIN_KEYS).reset_index(drop=True)
    return df


def merge_est_meas(est_df, meas_df) -> pd.DataFrame:
    merged = pd.merge(est_df, meas_df, on=JOIN_KEYS, suffixes=("_est", "_meas"))
    return merged.sort_values(JOIN_KEYS).reset_index(drop=True)


def compute_mape(merged_df) -> pd.DataFrame:
    def _mape(g):
        return pd.Series({
            f"{c}_mape%": ((g[f"{c}_est"] - g[f"{c}_meas"]).abs() / g[f"{c}_meas"].abs().clip(lower=1e-9) * 100).mean()
            for c in COLS
        })
    return merged_df.groupby(["tp", "pp"]).apply(_mape).round(2).reset_index()


def compute_overall_mape(merged_df) -> pd.Series:
    result = pd.Series({
        f"{c}_mape%": ((merged_df[f"{c}_est"] - merged_df[f"{c}_meas"]).abs() / merged_df[f"{c}_meas"].abs().clip(lower=1e-9) * 100).mean()
        for c in COLS
    })
    return result.round(2)


def max_batch_summary(df):
    return (
        df.loc[df.groupby(["tp", "pp"])["batch"].idxmax()]
        .sort_values(["tp", "pp"])
        .reset_index(drop=True)
    )

---
## 1. Estimated

In [ ]:
est_df = load_predicted(PRED_DIR, INSTANCE)
print(f"Estimated: {len(est_df)} rows, {est_df.groupby(['tp','pp']).ngroups} configs")
max_batch_summary(est_df)

---
## 2. Measured

In [ ]:
meas_df = load_measured(MEAS_DIR)
if meas_df.empty:
    print("No measured data yet.")
else:
    meas_with_mape = pd.merge(meas_df, est_df, on=JOIN_KEYS, suffixes=('', '_est'))
    for c in COLS:
        meas_with_mape[f"{c}_mape%"] = ((meas_with_mape[f"{c}_est"] - meas_with_mape[c]).abs() / meas_with_mape[c].abs().clip(lower=1e-9) * 100).round(2)
    drop_cols = [f"{c}_est" for c in COLS]
    meas_with_mape = meas_with_mape.drop(columns=drop_cols)
    print(f"Measured: {len(meas_df)} rows, {meas_df.groupby(['tp','pp']).ngroups} configs")
    display(max_batch_summary(meas_with_mape))

---
## 3. Estimated vs Measured

In [ ]:
if not meas_df.empty:
    merged = merge_est_meas(est_df, meas_df)
    print(f"Matched: {len(merged)} rows")
    display(merged)
else:
    print("No measured data to compare.")

### 3a. MAPE per Config (%)

In [ ]:
if not meas_df.empty:
    display(compute_mape(merged))

### 3b. Overall MAPE

In [ ]:
if not meas_df.empty:
    overall = compute_overall_mape(merged)
    print("Overall MAPE:")
    for k, v in overall.items():
        print(f"  {k}: {v}%")

### 3c. Config Summary (all batch sizes, with MAPE)

In [ ]:
if not meas_df.empty:
    summary = merged[["tp", "pp", "batch"]].copy()
    for c in COLS:
        summary[f"{c}_est"] = merged[f"{c}_est"].values
        summary[f"{c}_meas"] = merged[f"{c}_meas"].values
        meas_vals = merged[f"{c}_meas"].values
        summary[f"{c}_mape%"] = (abs(merged[f"{c}_est"].values - meas_vals) / abs(meas_vals).clip(min=1e-9) * 100).round(2)
    display(summary)

---
## 4. Estimated vs Measured Bar Charts

Per strategy (tp, pp), across all batch sizes.  
Left: raw values. Right: normalized to batch=1.

In [ ]:
PLOT_COLS = ["e2e_ms", "rps"]
PLOT_LABELS = {"e2e_ms": "E2E Latency (ms)", "rps": "Throughput (rps)"}
NORM_LABELS = {"e2e_ms": "E2E (norm)", "rps": "RPS (norm)"}
C_EST, C_MEAS = "#4C72B0", "#DD8452"

if not meas_df.empty:
    strat_groups = list(merged.groupby(["tp", "pp"]))
    n_strats = len(strat_groups)

    fig, axes = plt.subplots(n_strats, 4, figsize=(22, 4 * n_strats))
    if n_strats == 1:
        axes = [axes]

    fig.legend(
        handles=[Patch(color=C_EST, label="Estimated"), Patch(color=C_MEAS, label="Measured (TRT-LLM)")],
        loc="upper center", ncol=2, fontsize=12, frameon=False,
        bbox_to_anchor=(0.5, 1.02),
    )
    fig.suptitle(f"{INSTANCE_MAP.get(INSTANCE, INSTANCE)}", fontsize=15, fontweight="bold", y=1.05)

    for row, ((tp, pp), grp) in enumerate(strat_groups):
        grp = grp.sort_values("batch")
        batches = grp["batch"].values
        x = np.arange(len(batches))
        w = 0.35

        for ci, c in enumerate(PLOT_COLS):
            est_vals = grp[f"{c}_est"].values.astype(float)
            meas_vals = grp[f"{c}_meas"].values.astype(float)
            mape_vals = (abs(est_vals - meas_vals) / abs(meas_vals).clip(min=1e-9) * 100)

            # Raw
            ax_raw = axes[row][ci]
            be = ax_raw.bar(x - w/2, est_vals, w, color=C_EST)
            bm = ax_raw.bar(x + w/2, meas_vals, w, color=C_MEAS)
            for i, mp in enumerate(mape_vals):
                top = max(be[i].get_height(), bm[i].get_height())
                ax_raw.text(i, top * 1.02, f"{mp:.0f}%", ha="center", va="bottom", fontsize=8, color="#555")
            ax_raw.set_title(f"tp{tp}_pp{pp} — {PLOT_LABELS[c]}", fontsize=11)
            ax_raw.set_xticks(x)
            ax_raw.set_xticklabels([str(b) for b in batches], fontsize=8)
            ax_raw.set_xlabel("batch size")
            ax_raw.set_ylim(0, ax_raw.get_ylim()[1] * 1.18)

            # Normalized to batch=1
            ax_norm = axes[row][ci + 2]
            base_est = est_vals[0] if est_vals[0] != 0 else 1
            base_meas = meas_vals[0] if meas_vals[0] != 0 else 1
            norm_est = est_vals / base_est
            norm_meas = meas_vals / base_meas
            norm_mape = abs(norm_est - norm_meas) / abs(norm_meas).clip(min=1e-9) * 100
            be2 = ax_norm.bar(x - w/2, norm_est, w, color=C_EST)
            bm2 = ax_norm.bar(x + w/2, norm_meas, w, color=C_MEAS)
            for i, d in enumerate(norm_mape):
                top = max(be2[i].get_height(), bm2[i].get_height())
                ax_norm.text(i, top * 1.02, f"{d:.0f}%", ha="center", va="bottom", fontsize=8, color="#555")
            ax_norm.set_title(f"tp{tp}_pp{pp} — {NORM_LABELS[c]}", fontsize=11)
            ax_norm.set_xticks(x)
            ax_norm.set_xticklabels([str(b) for b in batches], fontsize=8)
            ax_norm.set_xlabel("batch size")
            ax_norm.axhline(y=1, color="gray", linestyle="--", linewidth=0.8)
            ax_norm.set_ylim(0, ax_norm.get_ylim()[1] * 1.12)

    plt.tight_layout()
    plt.show()

---
## 5. Filter

In [ ]:
def filter_df(df, tp=None, pp=None):
    if tp:
        df = df[df["tp"] == tp]
    if pp:
        df = df[df["pp"] == pp]
    return df.reset_index(drop=True)

# Example: filter_df(summary, tp=8, pp=1)